# Full Sales Forecasting Pipeline Template

This notebook provides a ready-to-run template for a complete machine learning pipeline, from loading data to evaluating and saving a trained model.

Use this notebook as a starting point for your sales forecasting project.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')

In [ ]:
DATA_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')
MODEL_PATH = Path('../models/saved_model.pkl')
REPORT_PATH = Path('../reports/figures')

RANDOM_SEED = 42
TEST_SIZE = 0.2
TARGET_COLUMN = 'sales'

DATA_PATH, PROCESSED_PATH, MODEL_PATH, REPORT_PATH

In [ ]:
import os

csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('No CSV files found in ../data/raw. Place your dataset there first.')

raw_file = DATA_PATH / csv_files[0]
df = pd.read_csv(raw_file)
print(f'Loaded file: {raw_file}')
print(df.head())
print('\nShape:', df.shape)
print('\nColumns:', list(df.columns))

In [ ]:
# Basic cleaning example
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('string').fillna('missing')

# Fill numeric missing values
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

# Create a simple date feature if a date column exists
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek

# Save processed data
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
processed_file = PROCESSED_PATH / 'processed_data.csv'
df.to_csv(processed_file, index=False)
print(df.head())
print('\nProcessed data saved to', processed_file)

In [ ]:
# Quick exploratory visualization
if 'sales' in df.columns:
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df, x=df.index, y='sales')
    plt.title('Sales Trend')
    plt.xlabel('Row')
    plt.ylabel('Sales')
    plt.tight_layout()
    plt.show()
else:
    print('Target column "sales" not found. Update TARGET_COLUMN if your dataset uses a different name.')

In [ ]:
# Prepare features and target
feature_columns = [col for col in df.columns if col != TARGET_COLUMN]
X = df[feature_columns]
y = df[TARGET_COLUMN] if TARGET_COLUMN in df.columns else None

if y is None:
    raise KeyError(f'Target column {TARGET_COLUMN} not found. Adjust TARGET_COLUMN to match your dataset.')

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
model.fit(X_train, y_train)
print('Training complete')

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print('MAE:', round(mae, 4))
print('MSE:', round(mse, 4))
print('RMSE:', round(rmse, 4))
print('R2:', round(r2, 4))

plt.figure(figsize=(8, 4))
sns.scatterplot(x=y_test, y=y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Predicted vs Actual')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.tight_layout()
plt.show()

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

# Optional: save a simple metrics summary
metrics = {
    'mae': mae,
    'mse': mse,
    'rmse': rmse,
    'r2': r2,
}

pd.Series(metrics).to_csv(PROCESSED_PATH / 'model_metrics.csv')
print('Metrics saved to', PROCESSED_PATH / 'model_metrics.csv')

In [ ]:
# Run the full workflow end to end
# This cell executes the sequence of steps defined above in one place.

# Re-run configuration and load data
DATA_PATH = Path('../data/raw')
PROCESSED_PATH = Path('../data/processed')
MODEL_PATH = Path('../models/saved_model.pkl')
REPORT_PATH = Path('../reports/figures')

RANDOM_SEED = 42
TEST_SIZE = 0.2
TARGET_COLUMN = 'sales'

csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('No CSV files found in ../data/raw. Place your dataset there first.')

raw_file = DATA_PATH / csv_files[0]
df = pd.read_csv(raw_file)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('string').fillna('missing')

for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH / 'processed_data.csv', index=False)

feature_columns = [col for col in df.columns if col != TARGET_COLUMN]
X = df[feature_columns]
y = df[TARGET_COLUMN]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED)

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])

model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

metrics = {
    'mae': mean_absolute_error(y_test, y_pred),
    'mse': mean_squared_error(y_test, y_pred),
    'rmse': mean_squared_error(y_test, y_pred, squared=False),
    'r2': r2_score(y_test, y_pred),
}

print(metrics)
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')